# EDA

In [ ]:
import os
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns   
from src.aggregation.agg import ElectricityAggregator

In [2]:
# data paths
DATA_DIR = "mock_data"
OUT_BASE_DIR = "output"

MATDATA_PATH = os.path.join(DATA_DIR, "meter_data_500.snappy.parquet")
TARIFF_PATH  = os.path.join(DATA_DIR, "tariff_100.csv")
SURVEY_PATH  = os.path.join(DATA_DIR, "survey_50.csv")

# check dir
CHECK_DIR = os.path.join(OUT_BASE_DIR, "check")
os.makedirs(CHECK_DIR, exist_ok=True)

OUT_MAT_CHECK    = os.path.join(CHECK_DIR, "matdata_id_coverage_check.csv")
OUT_TARIFF_CHECK = os.path.join(CHECK_DIR, "tariff_check_vs_matdata.csv")

In [3]:
# Columns
COL_ID = "aID"
COL_TS = "TIDPUNKT"
COL_D  = "TIDPUNKT_DAG"
COL_VALUE = "FORBRUKNING_KWH"

T_COL_ID = "GS1-nr."
T_COL_SD = "Startdatum"


In [4]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(MATDATA_PATH)
print(pf.schema_arrow)

table = pf.read(columns=[COL_ID, COL_TS, COL_D, COL_VALUE])
meter_data = table.to_pandas()

aID: large_string
TIDPUNKT: timestamp[us, tz=UTC]
TIDPUNKT_DAG: date32[day]
FORBRUKNING_KWH: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 589


In [5]:
print(meter_data.head())
print(len(meter_data["aID"].unique())) # unique households

print(meter_data["TIDPUNKT_DAG"].unique()[:10]) 
print(len(meter_data["TIDPUNKT_DAG"].unique())) # unique days

                  aID                  TIDPUNKT TIDPUNKT_DAG  FORBRUKNING_KWH
0  735999166200000851 2024-01-01 00:00:00+00:00   2024-01-01         0.283425
1  735999166200000851 2024-01-01 01:00:00+00:00   2024-01-01         0.300204
2  735999166200000851 2024-01-01 02:00:00+00:00   2024-01-01         0.260809
3  735999166200000851 2024-01-01 03:00:00+00:00   2024-01-01         0.316894
4  735999166200000851 2024-01-01 04:00:00+00:00   2024-01-01         0.211205
500
[datetime.date(2024, 1, 1) datetime.date(2024, 1, 2)
 datetime.date(2024, 1, 3) datetime.date(2024, 1, 4)
 datetime.date(2024, 1, 5) datetime.date(2024, 1, 6)
 datetime.date(2024, 1, 7) datetime.date(2024, 1, 8)
 datetime.date(2024, 1, 9) datetime.date(2024, 1, 10)]
782


In [6]:
# survey & tariff
# survey_df = pd.read_csv(SURVEY_PATH)
tariff_df = pd.read_csv(TARIFF_PATH, sep=",")

print("=== number of samples ===")
# print(f"survey rows: {len(survey_df):,}") # Survey number
print(f"tariff rows: {len(tariff_df):,}") # Tariff number 

# display(survey_df.head(3))
display(tariff_df.head(3))



=== number of samples ===
tariff rows: 100


,Produktnamn,Startdatum,GS1-nr
0,GENAB Tidsindelad 6 kW Villa,2025-06-01,735999166200288372
1,GENAB Tidsindelad 6 kW Villa,2025-04-01,735999166200186244
2,GENAB Tidsindelad 6 kW Villa,2025-03-01,735999166200055594


# Electricity meter data 

In [9]:
agg = ElectricityAggregator(meter_data, tariff_df)

Initializing ElectricityAggregator...
Loaded 9,372,500 rows


In [10]:
result = agg.run(
    freq="month",
    agg_method=["variance", "top3_mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
)


Electricity aggregation start
Rows: 9,372,500
Frequency: month
Aggregation: ['variance', 'top3_mean']
Include price split (all/high/low): True
Add User groups: [0.25, 0.75]
Merging tariff data...
Tariff merge done (1.96s)
Creating usage groups...
Usage groups created (2.05s)
Aggregating per household...
Aggregation done (55.14s)
Total runtime: 59.15s


In [17]:
result.head()

,aID,TIDPUNKT,price,variance_consumption,top3_mean_consumption,tariff_start,tariff_plan,tariff_active,usage_group
0,735999166200000851,2024-01-01,all,0.022423,0.774499,NaT,NaN,0,low
1,735999166200000851,2024-02-01,all,0.023686,0.784520,NaT,NaN,0,low
2,735999166200000851,2024-03-01,all,0.023351,0.787251,NaT,NaN,0,low
3,735999166200000851,2024-04-01,all,0.017125,0.645235,NaT,NaN,0,low
4,735999166200000851,2024-05-01,all,0.017482,0.645829,NaT,NaN,0,low


# tariff